In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
from pathlib import Path

In [ ]:
import pandas as pd

df_spk = pd.read_excel("/content/drive/MyDrive/TP1/data/Quechua_Collao_Corpus/Data/Data/Data.xlsx", sheet_name="map")
df_spk.head()

,Audio,Emotion,Actor,File,Duration (s)
0,10001,anger,a2,a2_A069.wav,1.916009
1,10002,boredom,a2,a2_B159.wav,7.210522
2,10003,anger,a2,a2_A211.wav,9.036009
3,10004,boredom,a5,a5_B221.wav,5.563991
4,10005,happy,a2,a2_H089.wav,4.060000


In [ ]:
UNIFIED = Path("/content/drive/MyDrive/TP1/data/unificado.csv")

In [ ]:
df = pd.read_csv(UNIFIED)
df.head()

,path,corpus,lang,speaker_id,clip_id,valence,arousal,dominance
0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,quechua,qu,21421,21421-3.25-2.5-2.75,3.25,2.50,2.75
1,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,quechua,qu,21422,21422-1.75-3.5-4,1.75,3.50,4.00
2,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,quechua,qu,21423,21423-2.75-2.5-2.75,2.75,2.50,2.75
3,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,quechua,qu,21424,21424-3.25-3-2.75,3.25,3.00,2.75
4,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,quechua,qu,21425,21425-3.25-2.25-1.75,3.25,2.25,1.75


In [ ]:
print("df_spk columns:", df_spk.columns.tolist())
print("df columns:", df.columns.tolist())

df_spk columns: ['Audio', 'Emotion', 'Actor', 'File', 'Duration (s)']
df columns: ['path', 'corpus', 'lang', 'speaker_id', 'clip_id', 'valence', 'arousal', 'dominance']


In [ ]:
def get_id_from_clip(clip_id):
    return str(clip_id).split("-")[0]

df["id_audio"] = df["speaker_id"].astype(str).apply(get_id_from_clip)

In [ ]:
df_spk = df_spk.rename(columns={
    "Audio": "id_audio",
    "Actor": "speaker_qu"
})

In [ ]:
df_spk["id_audio"] = df_spk["id_audio"].astype(str)
df["id_audio"] = df["id_audio"].astype(str)

In [ ]:
df = df.merge(
    df_spk[["id_audio", "speaker_qu"]],
    on="id_audio",
    how="left"
)

In [ ]:
df["speaker_id_final"] = df["speaker_id"].astype(str)

mask_qu = df["lang"] == "qu"
df.loc[mask_qu, "speaker_id_final"] = df.loc[mask_qu, "speaker_qu"]


In [ ]:
print("\nSpeakers por idioma:")
print(df.groupby("lang")["speaker_id_final"].nunique(dropna=False))

print("\nEjemplo quechua:")
print(df[df["lang"] == "qu"][["clip_id", "speaker_id", "speaker_qu", "speaker_id_final"]].head())

print("\nFilas quechua sin speaker mapeado:")
print(df[(df["lang"] == "qu") & (df["speaker_id_final"].isna())][["clip_id", "id_audio"]].head(20))


Speakers por idioma:
lang
es    14
qu     8
Name: speaker_id_final, dtype: int64

Ejemplo quechua:
                clip_id speaker_id speaker_qu speaker_id_final
0   21421-3.25-2.5-2.75      21421         a3               a3
1      21422-1.75-3.5-4      21422         a4               a4
2   21423-2.75-2.5-2.75      21423         a4               a4
3     21424-3.25-3-2.75      21424         a1               a1
4  21425-3.25-2.25-1.75      21425         a4               a4

Filas quechua sin speaker mapeado:
Empty DataFrame
Columns: [clip_id, id_audio]
Index: []


In [ ]:
df[df['speaker_id_final']=='6-']

,path,corpus,lang,speaker_id,clip_id,valence,arousal,dominance,id_audio,speaker_qu,speaker_id_final
8776,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,quechua,qu,14197,14197-3-2.25-2.25,3.0,2.25,2.25,14197,6-,6-


In [ ]:
df[df['speaker_id_final']=='2_']

,path,corpus,lang,speaker_id,clip_id,valence,arousal,dominance,id_audio,speaker_qu,speaker_id_final
1467,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,quechua,qu,20888,20888-3.25-3-3,3.25,3.0,3.0,20888,2_,2_


In [ ]:
# quitar speakers mal mapeados
bad_speakers = ["2_", "6-"]

df = df[~df["speaker_id_final"].astype(str).isin(bad_speakers)].copy()
df = df.reset_index(drop=True)

# verificar
print(df.groupby("lang")["speaker_id_final"].nunique(dropna=False))
print(df[df["speaker_id_final"].astype(str).isin(bad_speakers)].shape)

lang
es    14
qu     6
Name: speaker_id_final, dtype: int64
(0, 11)


In [ ]:
df["group_id"] = df["lang"].astype(str) + "_" + df["speaker_id_final"].astype(str)

In [ ]:
df

,path,corpus,lang,speaker_id,clip_id,valence,arousal,dominance,id_audio,speaker_qu,speaker_id_final,group_id
0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,quechua,qu,21421,21421-3.25-2.5-2.75,3.25,2.50,2.75,21421,a3,a3,qu_a3
1,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,quechua,qu,21422,21422-1.75-3.5-4,1.75,3.50,4.00,21422,a4,a4,qu_a4
2,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,quechua,qu,21423,21423-2.75-2.5-2.75,2.75,2.50,2.75,21423,a4,a4,qu_a4
3,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,quechua,qu,21424,21424-3.25-3-2.75,3.25,3.00,2.75,21424,a1,a1,qu_a1
4,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,quechua,qu,21425,21425-3.25-2.25-1.75,3.25,2.25,1.75,21425,a4,a4,qu_a4
...,...,...,...,...,...,...,...,...,...,...,...,...
16176,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio9,Audio9_83-03-03-04,3.00,3.00,4.00,Audio9,NaN,Audio9,es_Audio9
16177,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio9,Audio9_99-02-02-04,2.00,2.00,4.00,Audio9,NaN,Audio9,es_Audio9
16178,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio9,Audio9_86-03-03-04,3.00,3.00,4.00,Audio9,NaN,Audio9,es_Audio9
16179,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio9,Audio9_89-03-03-04,3.00,3.00,4.00,Audio9,NaN,Audio9,es_Audio9


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# 70/15/15 aprox
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
idx_trval, idx_te = next(gss1.split(df, groups=df["group_id"]))

df_trval = df.iloc[idx_trval].copy()
df_te    = df.iloc[idx_te].copy()

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=42)
idx_tr, idx_va = next(gss2.split(df_trval, groups=df_trval["group_id"]))

df_tr = df_trval.iloc[idx_tr].copy()
df_va = df_trval.iloc[idx_va].copy()

In [ ]:
spk_tr = set(df_tr["group_id"])
spk_va = set(df_va["group_id"])
spk_te = set(df_te["group_id"])

print("TR∩VA:", spk_tr & spk_va)
print("TR∩TE:", spk_tr & spk_te)
print("VA∩TE:", spk_va & spk_te)

TR∩VA: set()
TR∩TE: set()
VA∩TE: set()


In [ ]:
for name, d in [("train", df_tr), ("val", df_va), ("test", df_te)]:
    print(f"\n{name}")
    print(d.groupby("lang")["speaker_id_final"].nunique())
    print(d["lang"].value_counts())


train
lang
es    10
qu     3
Name: speaker_id_final, dtype: int64
lang
qu    6209
es    2624
Name: count, dtype: int64

val
lang
es    3
qu    1
Name: speaker_id_final, dtype: int64
lang
qu    2028
es     501
Name: count, dtype: int64

test
lang
es    1
qu    2
Name: speaker_id_final, dtype: int64
lang
qu    4181
es     638
Name: count, dtype: int64


In [ ]:
for name, d in [("train", df_tr), ("val", df_va), ("test", df_te)]:
    print(f"\n{name}")
    print(d[["valence","arousal","dominance"]].describe())


train
           valence      arousal    dominance
count  8833.000000  8833.000000  8833.000000
mean      2.988518     3.204857     3.374326
std       0.954227     0.899277     0.962121
min       1.000000     1.000000     1.000000
25%       2.250000     2.500000     2.750000
50%       3.000000     3.000000     3.250000
75%       3.750000     4.000000     4.000000
max       5.000000     5.000000     5.000000

val
           valence      arousal    dominance
count  2529.000000  2529.000000  2529.000000
mean      3.031831     3.037861     3.102906
std       0.946280     0.895816     0.845901
min       1.000000     1.000000     1.000000
25%       2.250000     2.250000     2.500000
50%       3.250000     3.000000     3.000000
75%       3.750000     3.750000     3.750000
max       5.000000     5.000000     5.000000

test
           valence      arousal    dominance
count  4819.000000  4819.000000  4819.000000
mean      3.084976     3.065263     3.176645
std       0.673942     0.795816     0

In [ ]:
def stats_split(df, name):
    print(f"\n{name}")
    for col in ["valence","arousal","dominance"]:
        print(col, "mean:", df[col].mean(), "std:", df[col].std())

stats_split(df_tr, "train")
stats_split(df_va, "val")
stats_split(df_te, "test")


train
valence mean: 2.9885180572851806 std: 0.9542270981138061
arousal mean: 3.204856787048568 std: 0.8992773772084495
dominance mean: 3.3743258236159854 std: 0.9621213146849874

val
valence mean: 3.031830763147489 std: 0.9462801137221672
arousal mean: 3.037860814551206 std: 0.8958164296084704
dominance mean: 3.102906287069988 std: 0.8459008892268651

test
valence mean: 3.0849761361278274 std: 0.6739423354672192
arousal mean: 3.065262502593899 std: 0.7958158981441298
dominance mean: 3.1766445320605934 std: 0.8315238356072305


In [ ]:
for col in ["valence", "arousal", "dominance"]:
    print("\n", col)
    print("train")
    print(df_tr[col].value_counts().sort_index())
    print("val")
    print(df_va[col].value_counts().sort_index())
    print("test")
    print(df_te[col].value_counts().sort_index())


 valence
train
valence
1.00     528
1.25      90
1.50     143
1.75     279
2.00     850
2.25     422
2.33       1
2.50     533
2.75     550
3.00    1663
3.25     728
3.50     650
3.75     544
4.00     930
4.25     365
4.50     263
4.75     149
5.00     145
Name: count, dtype: int64
val
valence
1.00    100
1.25     38
1.50     57
1.75     97
2.00    301
2.25    102
2.50    138
2.75    157
3.00    259
3.25    238
3.50    254
3.75    193
4.00    326
4.25    135
4.50     95
4.75     28
5.00     11
Name: count, dtype: int64
test
valence
1.00       1
1.25      28
1.50      64
1.75     123
2.00     217
2.25     287
2.50     402
2.75     476
3.00    1052
3.25     601
3.50     508
3.75     417
4.00     341
4.25     210
4.50      78
4.75      12
5.00       2
Name: count, dtype: int64

 arousal
train
arousal
1.00      42
1.25      67
1.50     149
1.75     186
2.00     954
2.25     401
2.50     421
2.75     571
3.00    1992
3.25     551
3.50     547
3.75     520
4.00    1051
4.25     357
4.50    

In [ ]:
import pandas as pd

bins = [0, 2.5, 3.5, 5.01]
labels = ["low", "mid", "high"]

def show_bins(df, name):
    print(f"\n{name}")
    for col in ["valence","arousal","dominance"]:
        b = pd.cut(df[col], bins=bins, labels=labels, include_lowest=True)
        print(col)
        print((b.value_counts(normalize=True)*100).round(1).sort_index())

show_bins(df_tr, "train")
show_bins(df_va, "val")
show_bins(df_te, "test")


train
valence
valence
low     32.2
mid     40.7
high    27.1
Name: proportion, dtype: float64
arousal
arousal
low     25.1
mid     41.4
high    33.4
Name: proportion, dtype: float64
dominance
dominance
low     24.8
mid     31.8
high    43.4
Name: proportion, dtype: float64

val
valence
valence
low     32.9
mid     35.9
high    31.2
Name: proportion, dtype: float64
arousal
arousal
low     34.6
mid     38.2
high    27.2
Name: proportion, dtype: float64
dominance
dominance
low     30.7
mid     39.5
high    29.8
Name: proportion, dtype: float64

test
valence
valence
low     23.3
mid     54.7
high    22.0
Name: proportion, dtype: float64
arousal
arousal
low     30.2
mid     44.0
high    25.8
Name: proportion, dtype: float64
dominance
dominance
low     28.2
mid     41.5
high    30.3
Name: proportion, dtype: float64


In [ ]:
df_tr = df_tr.reset_index(drop=True)
df_va = df_va.reset_index(drop=True)
df_te = df_te.reset_index(drop=True)

In [ ]:
base = "/content/drive/MyDrive/pruebas/audio_speaker/splits/"

df_tr.to_csv(base + "train.csv", index=False)
df_va.to_csv(base + "val.csv", index=False)
df_te.to_csv(base + "test.csv", index=False)

In [ ]:
df_es = df[df["lang"]=="es"].copy()
df_qu = df[df["lang"]=="qu"].copy()

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

def split_group(df, test_size=0.15, val_size=0.15, seed=42):
    gss1 = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    idx_trval, idx_te = next(gss1.split(df, groups=df["group_id"]))

    df_trval = df.iloc[idx_trval]
    df_te = df.iloc[idx_te]

    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_size/(1-test_size), random_state=seed)
    idx_tr, idx_va = next(gss2.split(df_trval, groups=df_trval["group_id"]))

    df_tr = df_trval.iloc[idx_tr]
    df_va = df_trval.iloc[idx_va]

    return df_tr, df_va, df_te

tr_es, va_es, te_es = split_group(df_es)
tr_qu, va_qu, te_qu = split_group(df_qu)

In [ ]:
df_tr2 = pd.concat([tr_es, tr_qu]).reset_index(drop=True)
df_va2 = pd.concat([va_es, va_qu]).reset_index(drop=True)
df_te2 = pd.concat([te_es, te_qu]).reset_index(drop=True)

In [ ]:
for name, d in [("train", df_tr2), ("val", df_va2), ("test", df_te2)]:
    print(f"\n{name}")
    print(d.groupby("lang")["speaker_id_final"].nunique())
    print(d["lang"].value_counts())


train
lang
es    9
qu    4
Name: speaker_id_final, dtype: int64
lang
qu    8278
es    2022
Name: count, dtype: int64

val
lang
es    2
qu    1
Name: speaker_id_final, dtype: int64
lang
qu    2070
es     312
Name: count, dtype: int64

test
lang
es    3
qu    1
Name: speaker_id_final, dtype: int64
lang
qu    2070
es    1429
Name: count, dtype: int64


In [ ]:
for col in ["valence", "arousal", "dominance"]:
    print("\n", col)
    print("train")
    print(df_tr2[col].value_counts().sort_index())
    print("val")
    print(df_va2[col].value_counts().sort_index())
    print("test")
    print(df_te2[col].value_counts().sort_index())


 valence
train
valence
1.00     515
1.25      99
1.50     173
1.75     309
2.00     781
2.25     528
2.50     708
2.75     798
3.00    1343
3.25    1096
3.50     975
3.75     783
4.00    1210
4.25     474
4.50     279
4.75      92
5.00     137
Name: count, dtype: int64
val
valence
1.00     78
1.25     39
1.50     50
1.75    139
2.00    350
2.25    180
2.33      1
2.50    207
2.75    196
3.00    256
3.25    185
3.50    169
3.75    149
4.00    101
4.25     78
4.50     99
4.75     86
5.00     19
Name: count, dtype: int64
test
valence
1.00      36
1.25      18
1.50      41
1.75      51
2.00     237
2.25     103
2.50     158
2.75     189
3.00    1375
3.25     286
3.50     268
3.75     222
4.00     286
4.25     158
4.50      58
4.75      11
5.00       2
Name: count, dtype: int64

 arousal
train
arousal
1.00      27
1.25      75
1.50     202
1.75     328
2.00     998
2.25     722
2.50     758
2.75     903
3.00    1860
3.25     771
3.50     679
3.75     692
4.00    1007
4.25     377
4.50     

In [ ]:
show_bins(df_tr2, "train")
show_bins(df_va2, "val")
show_bins(df_te2, "test")


train
valence
valence
low     30.2
mid     40.9
high    28.9
Name: proportion, dtype: float64
arousal
arousal
low     30.2
mid     40.9
high    28.9
Name: proportion, dtype: float64
dominance
dominance
low     30.7
mid     37.7
high    31.6
Name: proportion, dtype: float64

val
valence
valence
low     43.8
mid     33.8
high    22.3
Name: proportion, dtype: float64
arousal
arousal
low     15.7
mid     35.1
high    49.1
Name: proportion, dtype: float64
dominance
dominance
low     17.8
mid     39.4
high    42.8
Name: proportion, dtype: float64

test
valence
valence
low     18.4
mid     60.5
high    21.1
Name: proportion, dtype: float64
arousal
arousal
low     30.4
mid     48.6
high    21.0
Name: proportion, dtype: float64
dominance
dominance
low     21.1
mid     28.4
high    50.6
Name: proportion, dtype: float64


In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
# df con columnas: lang, group_id, valence, arousal, dominance

def split_group(df_sub, test_size=0.15, val_size=0.15, seed=42):
    gss1 = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    idx_trval, idx_te = next(gss1.split(df_sub, groups=df_sub["group_id"]))

    df_trval = df_sub.iloc[idx_trval].copy()
    df_te = df_sub.iloc[idx_te].copy()

    val_size_rel = val_size / (1 - test_size)
    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_size_rel, random_state=seed)
    idx_tr, idx_va = next(gss2.split(df_trval, groups=df_trval["group_id"]))

    df_tr = df_trval.iloc[idx_tr].copy()
    df_va = df_trval.iloc[idx_va].copy()

    return (
        df_tr.reset_index(drop=True),
        df_va.reset_index(drop=True),
        df_te.reset_index(drop=True),
    )

def balance_score(df_tr, df_va, df_te):
    cols = ["valence", "arousal", "dominance"]
    score = 0.0

    for c in cols:
        score += abs(df_tr[c].mean() - df_va[c].mean())
        score += abs(df_tr[c].mean() - df_te[c].mean())
        score += abs(df_va[c].mean() - df_te[c].mean())

    return float(score)

def speaker_check(df_tr, df_va, df_te):
    a = set(df_tr["group_id"])
    b = set(df_va["group_id"])
    c = set(df_te["group_id"])
    return (a & b, a & c, b & c)

best_score = 1e9
best_split = None
all_results = []

df_es = df[df["lang"] == "es"].copy()
df_qu = df[df["lang"] == "qu"].copy()

for seed in range(50):
    try:
        # split por idioma
        tr_es, va_es, te_es = split_group(df_es, test_size=0.15, val_size=0.15, seed=seed)
        tr_qu, va_qu, te_qu = split_group(df_qu, test_size=0.15, val_size=0.15, seed=seed)

        # unir
        df_tr2 = pd.concat([tr_es, tr_qu], ignore_index=True)
        df_va2 = pd.concat([va_es, va_qu], ignore_index=True)
        df_te2 = pd.concat([te_es, te_qu], ignore_index=True)

        # verificar que no haya fuga
        overlap_tr_va, overlap_tr_te, overlap_va_te = speaker_check(df_tr2, df_va2, df_te2)
        if overlap_tr_va or overlap_tr_te or overlap_va_te:
            continue

        # score de balance
        score = balance_score(df_tr2, df_va2, df_te2)

        # guardar también info útil
        result = {
            "seed": seed,
            "score": score,
            "train_es_spk": df_tr2[df_tr2["lang"] == "es"]["speaker_id_final"].nunique(),
            "train_qu_spk": df_tr2[df_tr2["lang"] == "qu"]["speaker_id_final"].nunique(),
            "val_es_spk": df_va2[df_va2["lang"] == "es"]["speaker_id_final"].nunique(),
            "val_qu_spk": df_va2[df_va2["lang"] == "qu"]["speaker_id_final"].nunique(),
            "test_es_spk": df_te2[df_te2["lang"] == "es"]["speaker_id_final"].nunique(),
            "test_qu_spk": df_te2[df_te2["lang"] == "qu"]["speaker_id_final"].nunique(),
        }
        all_results.append(result)

        if score < best_score:
            best_score = score
            best_split = (
                df_tr2.copy(),
                df_va2.copy(),
                df_te2.copy(),
                seed
            )

    except Exception as e:
        print(f"Seed {seed} falló: {e}")

print("Best score:", best_score)
print("Best seed:", best_split[3] if best_split is not None else None)

results_df = pd.DataFrame(all_results).sort_values("score").reset_index(drop=True)
print(results_df.head(10))

Best score: 0.6952795938191834
Best seed: 36
   seed     score  train_es_spk  train_qu_spk  val_es_spk  val_qu_spk  \
0    36  0.695280             9             4           2           1   
1    49  0.769688             9             4           2           1   
2     3  0.868870             9             4           2           1   
3    23  0.885847             9             4           2           1   
4    12  0.955950             9             4           2           1   
5    43  1.026140             9             4           2           1   
6    25  1.069426             9             4           2           1   
7    27  1.071369             9             4           2           1   
8     7  1.160601             9             4           2           1   
9     9  1.185590             9             4           2           1   

   test_es_spk  test_qu_spk  
0            3            1  
1            3            1  
2            3            1  
3            3            1  
4

In [ ]:
df_tr_best, df_va_best, df_te_best, best_seed = best_split

In [ ]:
print("Best seed:", best_seed)

for name, d in [("train", df_tr_best), ("val", df_va_best), ("test", df_te_best)]:
    print(f"\n{name}")
    print(d.groupby("lang")["speaker_id_final"].nunique())
    print(d["lang"].value_counts())
    print(d[["valence", "arousal", "dominance"]].mean())

Best seed: 36

train
lang
es    9
qu    4
Name: speaker_id_final, dtype: int64
lang
qu    8278
es    1793
Name: count, dtype: int64
valence      3.038510
arousal      3.156464
dominance    3.197564
dtype: float64

val
lang
es    2
qu    1
Name: speaker_id_final, dtype: int64
lang
qu    2028
es     917
Name: count, dtype: int64
valence      3.044992
arousal      3.107555
dominance    3.405518
dtype: float64

test
lang
es    3
qu    1
Name: speaker_id_final, dtype: int64
lang
qu    2112
es    1053
Name: count, dtype: int64
valence      2.958373
arousal      3.103397
dominance    3.389889
dtype: float64


In [ ]:
show_bins(df_tr_best, "train")
show_bins(df_va_best, "val")
show_bins(df_te_best, "test")


train
valence
valence
low     32.1
mid     39.5
high    28.4
Name: proportion, dtype: float64
arousal
arousal
low     27.5
mid     41.3
high    31.1
Name: proportion, dtype: float64
dominance
dominance
low     27.4
mid     38.5
high    34.1
Name: proportion, dtype: float64

val
valence
valence
low     23.7
mid     54.0
high    22.3
Name: proportion, dtype: float64
arousal
arousal
low     26.9
mid     45.6
high    27.5
Name: proportion, dtype: float64
dominance
dominance
low     22.8
mid     33.1
high    44.1
Name: proportion, dtype: float64

test
valence
valence
low     27.5
mid     49.5
high    23.0
Name: proportion, dtype: float64
arousal
arousal
low     31.2
mid     39.3
high    29.5
Name: proportion, dtype: float64
dominance
dominance
low     28.3
mid     30.4
high    41.3
Name: proportion, dtype: float64


In [ ]:
df_tr_best = df_tr_best.reset_index(drop=True)
df_va_best = df_va_best.reset_index(drop=True)
df_te_best = df_te_best.reset_index(drop=True)

In [ ]:
df_tr_best.to_csv(base + "train_fin.csv", index=False)
df_va_best.to_csv(base + "val_fin.csv", index=False)
df_te_best.to_csv(base + "test_fin.csv", index=False)

In [ ]:
df_tr_best

,path,corpus,lang,speaker_id,clip_id,valence,arousal,dominance,id_audio,speaker_qu,speaker_id_final,group_id
0,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio10,Audio10_1-02-02-04,2.00,2.00,4.00,Audio10,NaN,Audio10,es_Audio10
1,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio10,Audio10_16-02-02-04,2.00,2.00,4.00,Audio10,NaN,Audio10,es_Audio10
2,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio10,Audio10_22-01-03-03,1.00,3.00,3.00,Audio10,NaN,Audio10,es_Audio10
3,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio10,Audio10_23-02-02-04,2.00,2.00,4.00,Audio10,NaN,Audio10,es_Audio10
4,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,globalv1,es,Audio10,Audio10_15-02-02-04,2.00,2.00,4.00,Audio10,NaN,Audio10,es_Audio10
...,...,...,...,...,...,...,...,...,...,...,...,...
10066,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,quechua,qu,10416,10416-3.5-2.75-2.5,3.50,2.75,2.50,10416,a6,a6,qu_a6
10067,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,quechua,qu,10417,10417-4.5-3.25-3.25,4.50,3.25,3.25,10417,a6,a6,qu_a6
10068,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,quechua,qu,10418,10418-4.25-3.5-3.75,4.25,3.50,3.75,10418,a1,a1,qu_a1
10069,/content/drive/.shortcut-targets-by-id/1Kj75Ss...,quechua,qu,10419,10419-2.25-3.75-3.5,2.25,3.75,3.50,10419,a2,a2,qu_a2


In [6]:
df_tr = pd.read_csv(base + "train_fin.csv")
df_va = pd.read_csv(base + "val_fin.csv")
df_te = pd.read_csv(base + "test_fin.csv")

In [7]:
df_tr['speaker_id_final'].value_counts()

,count
speaker_id_final,
a1,2070
a3,2070
a6,2069
a2,2069
Audio13,454
Audio5,230
Audio2,209
Audio3,199
Audio11,189


In [8]:
df_va['speaker_id_final'].value_counts()

,count
speaker_id_final,
a5,2028
Audio1,638
Audio9,279


In [9]:
df_te['speaker_id_final'].value_counts()

,count
speaker_id_final,
a4,2112
Audio7,561
Audio8,326
Audio6,166
